## **STEP 0 — Imports & setup**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## **STEP 1 — Load data from Excel**

In [ ]:
df = pd.read_excel("DataSet.xlsx")

src_texts = df["Arabic"].astype(str).tolist()
tgt_texts = df["English"].astype(str).tolist()


## **STEP 2 — Train / Validation / Test split**

In [ ]:
src_train, src_tmp, tgt_train, tgt_tmp = train_test_split(
    src_texts, tgt_texts, test_size=0.2, random_state=42
)

src_val, src_test, tgt_val, tgt_test = train_test_split(
    src_tmp, tgt_tmp, test_size=0.5, random_state=42
)


## **STEP 3 — Character-level vocabulary (TRAIN ONLY)**

In [ ]:
PAD, SOS, EOS, UNK = "<pad>", "<s>", "</s>", "<unk>"
def build_char_vocab(texts):
    chars = set()
    for t in texts:
        chars.update(list(t))
    vocab = [PAD, SOS, EOS, UNK] + sorted(chars)
    char2id = {c: i for i, c in enumerate(vocab)}
    id2char = {i: c for c, i in char2id.items()}
    return char2id, id2char



In [ ]:
src_char2id, src_id2char = build_char_vocab(src_train)
tgt_char2id, tgt_id2char = build_char_vocab(tgt_train)


## **STEP 4 — Encoding function**

In [ ]:
MAX_LEN = 500
def encode(text, char2id):
    ids = [char2id[SOS]]
    for c in text:
        ids.append(char2id.get(c, char2id[UNK]))
    ids.append(char2id[EOS])
    ids = ids[:MAX_LEN]
    ids += [char2id[PAD]] * (MAX_LEN - len(ids))
    return ids


## **STEP 5 — Dataset & DataLoaders**

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts):
        self.src = src_texts
        self.tgt = tgt_texts

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode(self.src[idx], src_char2id)),
            torch.tensor(encode(self.tgt[idx], tgt_char2id)),
        )


In [ ]:
train_loader = DataLoader(
    TranslationDataset(src_train, tgt_train),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    TranslationDataset(src_val, tgt_val),
    batch_size=32
)


## **STEP 6 — Transformer model (from scratch)**

In [ ]:
class CharTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, nhead=8, num_layers=4):
        super().__init__()

        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt):
        src = self.src_emb(src)
        tgt = self.tgt_emb(tgt)

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1)
        ).to(tgt.device)

        out = self.transformer(src, tgt, tgt_mask=tgt_mask)
        return self.fc(out)


## **STEP 7 — Training loop + validation**

In [ ]:
model = CharTransformer(
    len(src_char2id),
    len(tgt_char2id)
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=tgt_char2id[PAD])
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for src, tgt in train_loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        logits = model(src, tgt_in)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train loss: {train_loss:.3f}")


Epoch 1 | Train loss: 2499.806


## **STEP 8 — Save model + vocabularies**

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "src_char2id": src_char2id,
    "tgt_char2id": tgt_char2id,
    "src_id2char": src_id2char,
    "tgt_id2char": tgt_id2char
}, "/content/drive/MyDrive/Run/Ar_En_transformer.pt")


## **STEP 9 — Load model for inference**

In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/Run/Ar_En_transformer.pt", map_location=DEVICE)

model.load_state_dict(checkpoint["model_state"])
model.to(DEVICE)
model.eval()

src_char2id = checkpoint["src_char2id"]
tgt_char2id = checkpoint["tgt_char2id"]
tgt_id2char = checkpoint["tgt_id2char"]


## **STEP 10 — Autoregressive inference**

In [ ]:
@torch.no_grad()
def translate(sentence):
    src = torch.tensor(
        encode(sentence, src_char2id)
    ).unsqueeze(0).to(DEVICE)

    tgt_ids = [tgt_char2id[SOS]]

    for _ in range(MAX_LEN):
        tgt = torch.tensor(tgt_ids).unsqueeze(0).to(DEVICE)
        logits = model(src, tgt)
        next_id = logits[0, -1].argmax().item()

        if next_id == tgt_char2id[EOS]:
            break
        tgt_ids.append(next_id)

    return "".join(tgt_id2char[i] for i in tgt_ids[1:])


## **STEP 11 — Translate the FULL test set**

In [ ]:
predictions = [translate(s) for s in src_test]

## **STEP 12 — Save outputs for evaluation**


In [ ]:
out_df = pd.DataFrame({
    "Arabic": src_test,
    "English_Reference": tgt_test,
    "English_Prediction": predictions
})


In [ ]:
out_df.to_excel("/content/drive/MyDrive/Run/Ar_En_predictions.xlsx", index=False)


In [ ]:
print("Done")

# **Evaluation**

## **STEP 0: Install Required Libraries (once)**

In [ ]:
pip install sacrebleu sentence-transformers unbabel-comet pandas torch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 39.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Succ

## **STEP 1: Load Translations from Excel**

In [ ]:
import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/Run/Ar_En_predictions.xlsx")

refs = df["English_Reference"].astype(str).tolist()
hyps = df["English_Prediction"].astype(str).tolist()

print("Samples:", len(refs))


Samples: 4744


## **STEP 2: BLEU (SacreBLEU – STANDARD)**

In [ ]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(hyps, [refs])
print("BLEU:", bleu.score)


## **STEP 3: chrF++ (BEST for Arabic)**

In [ ]:
chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2)
print("chrF++:", chrf.score)


## **STEP 4: SBERT Similarity (Semantic Score)**

In [ ]:
# For Araabic prediction evaluation
# from sentence_transformers import SentenceTransformer, util
# import numpy as np

# sbert = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# emb_hyp = sbert.encode(hyps, convert_to_tensor=True, show_progress_bar=True)
# emb_ref = sbert.encode(refs, convert_to_tensor=True, show_progress_bar=True)

# cos_scores = util.cos_sim(emb_hyp, emb_ref).diagonal()
# sbert_score = float(cos_scores.mean())

# print("SBERT cosine similarity:", sbert_score)


In [ ]:
# For Araabic prediction evaluation
from sentence_transformers import SentenceTransformer, util
import numpy as np

sbert = SentenceTransformer("all-MiniLM-L6-v2")

emb_hyp = sbert.encode(hyps, convert_to_tensor=True, show_progress_bar=True)
emb_ref = sbert.encode(refs, convert_to_tensor=True, show_progress_bar=True)

cos_scores = util.cos_sim(emb_hyp, emb_ref).diagonal()
sbert_score = float(cos_scores.mean())

print("SBERT cosine similarity:", sbert_score)


## **STEP 5: COMET (STATE-OF-THE-ART)**

### **5.1 Load Source Sentences**

In [ ]:
src = df["Arabic"].astype(str).tolist()


### **5.2 Load COMET Model**

In [ ]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

### **5.3 Prepare COMET Data**

In [ ]:
comet_data = [
    {"src": s, "mt": h, "ref": r}
    for s, h, r in zip(src, hyps, refs)
]


### **5.4 Run COMET Evaluation**

In [ ]:
comet_score = comet_model.predict(
    comet_data,
    batch_size=8,
    gpus=1 if comet_model.device.type == "cuda" else 0
)

print("COMET score:", comet_score.system_score)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
Predicting DataLoader 0: 100%|██████████| 593/593 [25:19<00:00,  2.56s/it]

COMET score: 0.7746948925228585


STEP 6: Save Metrics to Excel

In [ ]:
metrics_df = pd.DataFrame([{
    "BLEU": bleu.score,
    "chrF++": chrf.score,
    "SBERT": sbert_score,
    "COMET": comet_score.system_score
}])

metrics_df.to_excel("/content/drive/MyDrive/Run/Ar_En_evaluation_scores.xlsx", index=False)
metrics_df


,BLEU,chrF++,SBERT,COMET
0,3.210471,24.097096,0.784329,0.774695


In [ ]:
print("Done")

Done
